# Exercise 3.1: Dilepton tree and the Z peak

From *Programming in High Energy Particle Physics*, Chapter 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch03/ex03_1.ipynb)

Exercise 3.1 Dilepton tree and the Z peak Create a ROOT file with a TTree containing muon and electron information. Fill it with 50,000 simulated events (each with 0–4 electrons and 0–4 muons, with a Z&nbsp;→&nbsp;\(\ell\ell\) decay in part of the events as in Listing 3.6 ). Compute and plot the invariant mass of opposite-sign, same-flavor dilepton pairs. Can you see the Z boson peak around 91&nbsp;GeV?

<details><summary>Hint</summary>

Start from Listing 3.6 and change the lepton multiplicities. For the mass, loop over all pairs \((i, j)\) with \(i &lt; j\) of the same flavor, require charge[i]*charge[j] &lt; 0 , build ROOT::Math::PtEtaPhiMVector objects and fill (a + b).M() . Random leptons pair up too and form a smooth combinatorial background under the peak.

</details>

**Colab setup:** ROOT installation takes about 3–5 minutes and restarts the runtime once. After that restart, run all cells from the top again. The setup cells are safe to rerun.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q condacolab")
        import condacolab
        condacolab.install()  # one-time runtime restart

In [ ]:
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().system("mamba install -q -y -c conda-forge root")
        import ROOT
else:
    import ROOT
ROOT.gROOT.SetBatch(True)

### Generate a toy tree

Default: 8,000 events; set `N_EVENTS=50_000` for the full sample.

In [ ]:
import math
from array import array
from pathlib import Path
def create_tree(n_events=8000):
    rng=ROOT.TRandom3(42)
    f=ROOT.TFile("events.root","RECREATE")
    tree=ROOT.TTree("events","Toy dilepton events")
    floats={name:ROOT.std.vector('float')() for name in
        ('electron_pt','electron_eta','electron_phi','electron_iso',
         'muon_pt','muon_eta','muon_phi','muon_iso')}
    ints={name:ROOT.std.vector('int')() for name in ('electron_charge','muon_charge')}
    has_Z=array('i',[0]);z_pass=array('i',[0]);y_Z=array('f',[0.])
    for name,v in {**floats,**ints}.items():tree.Branch(name,v)
    tree.Branch('has_Z',has_Z,'has_Z/I');tree.Branch('z_pass',z_pass,'z_pass/I')
    tree.Branch('y_Z',y_Z,'y_Z/F')
    for _ in range(n_events):
        for v in [*floats.values(),*ints.values()]:v.clear()
        has_Z[0]=int(rng.Rndm()<0.35);z_pass[0]=0
        y_Z[0]=rng.Gaus(0,1.3) if has_Z[0] else 0.0
        for flavor in ('electron','muon'):
            for _ in range(rng.Integer(3)):
                pt=rng.Exp(25)+5;eta=rng.Gaus(0,1.8)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(rng.Uniform(-math.pi,math.pi))
                floats[flavor+'_iso'].push_back(rng.Exp(0.25))
                ints[flavor+'_charge'].push_back(1 if rng.Rndm()<0.5 else -1)
        if has_Z[0]:
            flavor='electron' if rng.Rndm()<0.5 else 'muon'
            mass=max(70.,rng.Gaus(91.1876,2.8))
            costh=rng.Uniform(-0.98,0.98);eta_star=math.atanh(costh)
            pt_true=0.5*mass*math.sqrt(1-costh*costh)
            z_pass[0]=int(pt_true>20 and abs(y_Z[0]+eta_star)<2.4
                          and abs(y_Z[0]-eta_star)<2.4)
            phi=rng.Uniform(-math.pi,math.pi)
            for charge,eta,azimuth in ((1,y_Z[0]+eta_star,phi),
                                       (-1,y_Z[0]-eta_star,phi+math.pi)):
                pt=pt_true*rng.Gaus(1,0.02)
                floats[flavor+'_pt'].push_back(pt)
                floats[flavor+'_eta'].push_back(eta)
                floats[flavor+'_phi'].push_back(azimuth)
                floats[flavor+'_iso'].push_back(rng.Exp(0.12/(1+pt/50)))
                ints[flavor+'_charge'].push_back(charge)
        tree.Fill()
    tree.Write();f.Close()
    print(f"Created {n_events} toy events in events.root")
N_EVENTS=8000  # set to 50_000 for the full exercise
create_tree(N_EVENTS)

### Build opposite-sign same-flavor masses

In [ ]:
import ROOT
# TODO: pair leptons with i < j and opposite charge using ROOT::Math::PtEtaPhiMVector.
# Histogram masses between 60 and 120 GeV.

In [ ]:
assert Path("events.root").exists()